# Day 38 — Greedy algorithms: interval scheduling and Huffman coding

A greedy algorithm makes the choice that looks best right now and never takes it
back. That is the whole idea, and it is why greedy code is short. The difficulty
is not writing it — it is knowing that the key you sorted by is the right one.

This is the part worth internalising before any of the code below: **a wrong
greedy key does not crash.** It returns a schedule with no clashes, a code that
decodes perfectly, a way to pay the exact amount. It is simply worse than the
best answer, and nothing in the program says so. Every section here pairs a
greedy with the ground truth so the gap is visible.

## 1. Eight people want the same meeting room

Eight requests, one room, half-open intervals: a meeting that ends at 11:00 does
not clash with one that starts at 11:00. Slots are 15 minutes from 09:00.

In [1]:
from itertools import combinations

T0 = 9 * 60          # the day starts at 09:00
UNIT = 15            # one slot is 15 minutes

def clock(u):
    m = T0 + u * UNIT
    return f'{m // 60:02d}:{m % 60:02d}'

def span(iv):
    return f'{clock(iv[0])}-{clock(iv[1])}'

MEETINGS = [
    ('A', 1, 9),      # 09:15-11:15   the long design review
    ('B', 6, 7),      # 10:30-10:45   standup
    ('C', 7, 8),      # 10:45-11:00   1:1
    ('D', 8, 12),     # 11:00-12:00   sprint planning
    ('E', 11, 13),    # 11:45-12:15   vendor call
    ('F', 12, 18),    # 12:00-13:30   lunch and learn
    ('G', 18, 23),    # 13:30-14:45   retro
    ('H', 18, 25),    # 13:30-15:15   customer demo
]
NAME = {(s, e): n for n, s, e in MEETINGS}
IV = [(s, e) for _, s, e in MEETINGS]

def names(chosen):
    return ' '.join(NAME[iv] for iv in chosen)

def compatible(a, b):
    """Half-open intervals clash only if neither ends before the other starts."""
    return a[1] <= b[0] or b[1] <= a[0]

def is_feasible(chosen):
    return all(compatible(x, y) for x, y in combinations(chosen, 2))

for n, s, e in MEETINGS:
    print(f'  {n}  {span((s, e))}  ({(e - s) * UNIT:>3d} min)')

  A  09:15-11:15  (120 min)
  B  10:30-10:45  ( 15 min)
  C  10:45-11:00  ( 15 min)
  D  11:00-12:00  ( 60 min)
  E  11:45-12:15  ( 30 min)
  F  12:00-13:30  ( 90 min)
  G  13:30-14:45  ( 75 min)
  H  13:30-15:15  (105 min)


## 2. The greedy: take whoever frees the room first

Sort by finish time, walk the list once, and accept a meeting whenever it starts
at or after the end of the last one accepted. One sort, one pass.

The reason this is the right key and not just *a* key: the only thing that
constrains everything still to come is the clock time at which the room becomes
free again. Picking the meeting that finishes earliest leaves the room free at
the earliest possible moment, so it can never close off an option that another
choice would have kept open.

In [2]:
def select_by_end(intervals):
    """Interval scheduling done right: always take the one that frees the room first."""
    out, last_end = [], float('-inf')
    for s, e in sorted(intervals, key=lambda iv: iv[1]):
        if s >= last_end:            # no clash with what we already took
            out.append((s, e))
            last_end = e
        # else: skip it - it overlaps something we already committed to
    return out

picked = select_by_end(IV)
for iv in picked:
    print(f'  take {NAME[iv]}  {span(iv)}')
print(f'  -> {len(picked)} meetings: {names(picked)}')

  take B  10:30-10:45
  take C  10:45-11:00
  take D  11:00-12:00
  take F  12:00-13:30
  take G  13:30-14:45
  -> 5 meetings: B C D F G


## 3. Two other keys that look just as reasonable

"Whoever asked first goes first" is how a human runs a room. "Pack the short
meetings first" is how you would fit socks into a drawer. Both are defensible.
Both are wrong, and the brute force over all 2⁸ subsets says by how much.

In [3]:
def select_by_start(intervals):
    """Plausible and wrong: whoever asked for the earliest slot goes first."""
    out, last_end = [], float('-inf')
    for s, e in sorted(intervals, key=lambda iv: iv[0]):
        if s >= last_end:
            out.append((s, e))
            last_end = e
    return out

def select_by_duration(intervals):
    """Also plausible and also wrong: pack the shortest meetings first."""
    out = []
    for s, e in sorted(intervals, key=lambda iv: (iv[1] - iv[0], iv[1])):
        if all(compatible((s, e), c) for c in out):
            out.append((s, e))
    return sorted(out)

def select_bruteforce(intervals):
    """Ground truth: the largest clash-free subset, found by trying every subset."""
    best = []
    n = len(intervals)
    for mask in range(1 << n):
        pick = [intervals[i] for i in range(n) if mask >> i & 1]
        if len(pick) > len(best) and is_feasible(pick):
            best = pick
    return sorted(best)

best = select_bruteforce(IV)
by_s = select_by_start(IV)
by_d = select_by_duration(IV)
print(f'  by end time   {len(picked)} meetings   {names(picked)}')
print(f'  by start time {len(by_s)} meetings   {names(by_s)}   <- A alone blocks 09:15-11:15')
print(f'  by duration   {len(by_d)} meetings   {names(by_d)}   <- the two 15-minute meetings look cheap')
print(f'  brute force   {len(best)} meetings   {names(best)}  (all 2^8 subsets)')
print()
print('  every one of the three is a legal, clash-free schedule:',
      is_feasible(picked), is_feasible(by_s), is_feasible(by_d))
print('  nothing raises, nothing warns - the wrong keys just seat fewer people.')

  by end time   5 meetings   B C D F G
  by start time 3 meetings   A E G   <- A alone blocks 09:15-11:15
  by duration   4 meetings   B C E G   <- the two 15-minute meetings look cheap
  brute force   5 meetings   B C D F G  (all 2^8 subsets)

  every one of the three is a legal, clash-free schedule: True True True
  nothing raises, nothing warns - the wrong keys just seat fewer people.


## 4. Why the end time is the right key: the exchange argument

The proof that a greedy is optimal almost always has the same shape. Claim: the
interval that finishes first belongs to *some* optimal schedule. Given any
optimal schedule that does not contain it, swap its first meeting for the
earliest finisher — the swap cannot create a clash, because the replacement ends
no later — and you have an optimal schedule of the same size that does contain it.

Below the claim is run as an experiment rather than argued: compare the best
schedule overall with the best schedule that is *forced* to contain the earliest
finisher. Equal sizes means the greedy choice costs nothing, and the same
argument then applies to whatever is left.

In [4]:
def exchange_argument(intervals):
    first = min(intervals, key=lambda iv: iv[1])
    unrestricted = len(select_bruteforce(intervals))
    rest = [iv for iv in intervals if compatible(iv, first) and iv != first]
    forced = 1 + len(select_bruteforce(rest))
    return first, unrestricted, forced

first, unrestricted, forced = exchange_argument(IV)
print(f'  earliest finishing meeting: {NAME[first]} {span(first)}')
print(f'  best schedule overall               : {unrestricted}')
print(f'  best schedule forced to contain {NAME[first]}   : {forced}')
print('  equal, so taking it first costs nothing.')

  earliest finishing meeting: B 10:30-10:45
  best schedule overall               : 5
  best schedule forced to contain B   : 5
  equal, so taking it first costs nothing.


## 5. Change the question slightly and the greedy dies

Now every meeting is worth money and we want the most valuable schedule rather
than the fullest one. The greedy still runs, still returns a real schedule, and
is now simply wrong: no sort key can work, because the value of taking a meeting
depends on what was given up for it. That needs a table — Day 33's DP, with
Day 23's `bisect` to find the last meeting that finishes before this one starts.

In [5]:
from bisect import bisect_right

VALUES = {
    (1, 9): 8, (6, 7): 1, (7, 8): 1, (8, 12): 3,
    (11, 13): 2, (12, 18): 4, (18, 23): 3, (18, 25): 5,
}

def greedy_weighted(intervals, value):
    """Run the same by-end greedy and just add up what it happens to collect."""
    chosen = select_by_end(intervals)
    return sum(value[iv] for iv in chosen), chosen

def dp_weighted(intervals, value):
    """Weighted interval scheduling needs a DP table, not a sort key."""
    iv = sorted(intervals, key=lambda t: t[1])
    ends = [e for _, e in iv]
    best = [0] * (len(iv) + 1)
    take = [False] * len(iv)
    for i, (s, e) in enumerate(iv):
        j = bisect_right(ends, s, 0, i)      # last compatible interval, 0 if none
        with_i = best[j] + value[(s, e)]
        best[i + 1] = max(best[i], with_i)
        take[i] = with_i > best[i]
    out, i = [], len(iv) - 1                 # walk the decisions backwards
    while i >= 0:
        s, e = iv[i]
        if take[i] and best[i + 1] != best[i]:
            out.append((s, e))
            i = bisect_right(ends, s, 0, i) - 1
        else:
            i -= 1
    return best[-1], sorted(out)

g_val, g_pick = greedy_weighted(IV, VALUES)
d_val, d_pick = dp_weighted(IV, VALUES)
print('  each meeting is now worth money:')
print('   ', '  '.join(f'{NAME[iv]}={VALUES[iv]}' for iv in IV))
print(f'  greedy by end time         : {g_val:>3d}  {names(g_pick)}')
print(f'  DP (Day 33 + Day 23 bisect): {d_val:>3d}  {names(d_pick)}')
print(f'  greedy leaves {d_val - g_val} on the table and still returns a real schedule.')

  each meeting is now worth money:
    A=8  B=1  C=1  D=3  E=2  F=4  G=3  H=5
  greedy by end time         :  12  B C D F G
  DP (Day 33 + Day 23 bisect):  17  A F H
  greedy leaves 5 on the table and still returns a real schedule.


## 6. Huffman coding — the same idea, and here it is provably optimal

A fixed-length code spends the same number of bits on the rarest symbol as on
the most common one. Huffman's insight is that the two rarest symbols can always
be siblings at the deepest level of the code tree, so the algorithm is: take the
two lightest subtrees off a min-heap, merge them, push the merged node back, and
repeat until one tree is left. Left branch is a `0`, right is a `1`.

That "push it back" is the whole algorithm. A merged node has a bigger weight
than either child and must re-enter the queue in its new place — which is what
makes the heap the right data structure rather than a sorted list.

The tie-breaker in the heap entry is not decoration: without it Python would
compare the node payloads when two weights are equal, so the code would depend
on how tuples of tuples happen to order.

In [6]:
import heapq
from collections import Counter

TEXT = 'greedy algorithms are easy to write and hard to prove'

def huffman(freqs):
    """Repeatedly merge the two rarest symbols.  Returns ({symbol: codeword}, merges)."""
    heap = [(w, i, sym) for i, (sym, w) in enumerate(sorted(freqs.items()))]
    heapq.heapify(heap)
    tie = len(heap)
    if len(heap) == 1:                    # a one-symbol alphabet still needs one bit
        return {heap[0][2]: '0'}, []
    merges = []
    while len(heap) > 1:
        w1, _, n1 = heapq.heappop(heap)
        w2, _, n2 = heapq.heappop(heap)
        merges.append((n1, n2, w1 + w2))
        heapq.heappush(heap, (w1 + w2, tie, (n1, n2)))   # back into the queue
        tie += 1
    root = heap[0][2]
    codes = {}
    def walk(node, prefix):
        if isinstance(node, str):
            codes[node] = prefix or '0'
            return
        walk(node[0], prefix + '0')
        walk(node[1], prefix + '1')
    walk(root, '')
    return codes, merges

def encoded_bits(freqs, codes):
    return sum(freqs[s] * len(codes[s]) for s in freqs)

def fixed_width_bits(freqs):
    """What a fixed-length code costs: ceil(log2(alphabet)) bits per symbol."""
    n = len(freqs)
    width = max(1, (n - 1).bit_length())
    return width, width * sum(freqs.values())

freqs = dict(Counter(TEXT))
codes, merges = huffman(freqs)
width, fixed = fixed_width_bits(freqs)
bits = encoded_bits(freqs, codes)
print(f'  text        : {TEXT!r}')
print(f'  {len(TEXT)} characters, {len(freqs)} distinct symbols')
print(f'  fixed-length: {width} bits each -> {fixed} bits')
print(f'  Huffman     : {bits} bits  ({100 * (fixed - bits) / fixed:.1f}% smaller)')
print('  first four merges (always the two rarest subtrees):')
for n1, n2, w in merges[:4]:
    print(f'    {str(n1):>20}  +  {str(n2):<20} = {w}')
print('  the five most common symbols get the shortest codewords:')
for sym, f in sorted(freqs.items(), key=lambda kv: -kv[1])[:5]:
    print(f'    {sym!r:>5} x{f:<3d} -> {codes[sym]}')

  text        : 'greedy algorithms are easy to write and hard to prove'
  53 characters, 18 distinct symbols
  fixed-length: 5 bits each -> 265 bits
  Huffman     : 204 bits  (23.0% smaller)
  first four merges (always the two rarest subtrees):
                       l  +  m                    = 2
                       n  +  p                    = 2
                       v  +  w                    = 2
                       g  +  h                    = 4
  the five most common symbols get the shortest codewords:
      ' ' x9   -> 111
      'r' x6   -> 011
      'e' x6   -> 010
      'a' x5   -> 001
      'o' x4   -> 1001


### Decoding needs no separators

No codeword is a prefix of another — that is what walking to a leaf guarantees —
so a decoder can read bits one at a time and emit a symbol the moment what it has
read is a codeword. There is no ambiguity to resolve and no delimiter to store.

The Kraft sum, Σ2^−len, is a cheap structural check: exactly 1.0 means the code
tree has no branch that leads nowhere. Below 1.0 means bits are being wasted on
a shape nothing uses.

In [7]:
def encode(text, codes):
    return ''.join(codes[c] for c in text)

def decode(bits, codes):
    back = {v: k for k, v in codes.items()}
    out, cur = [], ''
    for b in bits:
        cur += b
        if cur in back:
            out.append(back[cur])
            cur = ''
    assert cur == '', 'trailing bits: this was not a prefix code'
    return ''.join(out)

def is_prefix_free(codes):
    words = sorted(codes.values())
    return all(not b.startswith(a) for a, b in zip(words, words[1:]))

def kraft_sum(codes):
    """Sum of 2^-len.  Exactly 1.0 means the code tree has no wasted branch."""
    return sum(2.0 ** -len(c) for c in codes.values())

stream = encode(TEXT, codes)
print(f'  first 48 bits : {stream[:48]}')
print(f'  round-trips   : {decode(stream, codes) == TEXT}')
print(f'  prefix-free   : {is_prefix_free(codes)}')
print(f'  Kraft sum     : {kraft_sum(codes):.3f}')

  first 48 bits : 101100110100101000110101110011101101011010010111
  round-trips   : True
  prefix-free   : True
  Kraft sum     : 1.000


## 7. The silent failure: merging without re-sorting

Here is the bug worth remembering, because it is the one that produces no error.
Sort the symbols by frequency once, then merge left to right without ever putting
the merged node back into the queue. Every merge still joins two subtrees, so the
result is still a legal prefix code and still decodes the original text exactly.

What is lost is the re-sort. A merged node's weight has grown past its
neighbours and should sink back down the queue; here it never does, so it is
always one of the two lightest and the tree degenerates into a ladder. The
deepest leaf ends up with a codeword far longer than it deserves.

The file gets bigger. Nothing else changes.

In [8]:
def huffman_no_heap(freqs):
    """The bug: sort once by frequency and merge left to right, never re-sorting."""
    items = sorted(freqs.items(), key=lambda kv: (kv[1], kv[0]))
    node, weight = items[0][0], items[0][1]
    for sym, w in items[1:]:
        node, weight = ((node, sym), weight + w)
    codes = {}
    def walk(n, prefix):
        if isinstance(n, str):
            codes[n] = prefix or '0'
            return
        walk(n[0], prefix + '0')
        walk(n[1], prefix + '1')
    walk(node, '')
    return codes

naive = huffman_no_heap(freqs)
nbits = encoded_bits(freqs, naive)
print(f'  Huffman with a heap               : {bits} bits')
print(f'  sorted once, merged left to right : {nbits} bits  (+{nbits - bits}, {100 * (nbits - bits) / bits:.0f}% worse)')
print(f'  plain {width}-bit fixed-length code      : {fixed} bits  <- the broken version is worse than no compression at all')
print()
print(f'  prefix-free?  heap={is_prefix_free(codes)}   naive={is_prefix_free(naive)}')
print(f'  round-trips?  heap={decode(encode(TEXT, codes), codes) == TEXT}   naive={decode(encode(TEXT, naive), naive) == TEXT}')
print(f'  Kraft sum     heap={kraft_sum(codes):.3f}   naive={kraft_sum(naive):.3f}')
print('  both decode perfectly.  the broken one is just a bigger file.')
print(f'  the naive tree is a ladder: longest codeword {max(len(c) for c in naive.values())} bits'
      f' vs {max(len(c) for c in codes.values())}')

  Huffman with a heap               : 204 bits
  sorted once, merged left to right : 316 bits  (+112, 55% worse)
  plain 5-bit fixed-length code      : 265 bits  <- the broken version is worse than no compression at all

  prefix-free?  heap=True   naive=True
  round-trips?  heap=True   naive=True
  Kraft sum     heap=1.000   naive=1.000
  both decode perfectly.  the broken one is just a bigger file.
  the naive tree is a ladder: longest codeword 17 bits vs 6


## 8. The classic counterexample: coins 1, 3, 4

Paying 6 with coins of 1, 3 and 4: greedy takes the biggest coin that still fits,
so 4 + 1 + 1, three coins. Two coins of 3 would have done it. Greedy's answer is
a real way to pay — it just is not the fewest.

The reason nobody checks is that with the coins actually in a wallet (1, 5, 10,
25) the same greedy is always optimal. The denominations were chosen so that it
is, which is exactly what makes the habit dangerous when the denominations change.

In [9]:
COINS = [1, 3, 4]

def coin_greedy(coins, amount):
    left, used = amount, []
    for c in sorted(coins, reverse=True):
        while left >= c:
            left -= c
            used.append(c)
    return used if left == 0 else None

def coin_dp(coins, amount):
    best = [0] + [None] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and best[a - c] is not None:
                cand = best[a - c] + 1
                if best[a] is None or cand < best[a]:
                    best[a] = cand
    return best[amount]

for amount in (6, 8, 11):
    g = coin_greedy(COINS, amount)
    print(f'  pay {amount:>2d}: greedy {g} = {len(g)} coins,  optimal {coin_dp(COINS, amount)} coins')
print()
for amount in (6, 8, 11, 30, 63):
    g = coin_greedy([1, 5, 10, 25], amount)
    print(f'  pay {amount:>2d} with 1/5/10/25: greedy {len(g)} coins, optimal {coin_dp([1, 5, 10, 25], amount)} coins')

  pay  6: greedy [4, 1, 1] = 3 coins,  optimal 2 coins
  pay  8: greedy [4, 4] = 2 coins,  optimal 2 coins
  pay 11: greedy [4, 4, 3] = 3 coins,  optimal 3 coins

  pay  6 with 1/5/10/25: greedy 2 coins, optimal 2 coins
  pay  8 with 1/5/10/25: greedy 4 coins, optimal 4 coins
  pay 11 with 1/5/10/25: greedy 2 coins, optimal 2 coins
  pay 30 with 1/5/10/25: greedy 2 coins, optimal 2 coins
  pay 63 with 1/5/10/25: greedy 6 coins, optimal 6 coins


## 9. LeetCode 435 — Non-overlapping Intervals

Given a list of intervals, return the minimum number of them to remove so that
none of the survivors overlap.

This is section 2 with the answer inverted: maximising what is kept is the same
problem as minimising what is thrown away, so the by-finish-time greedy solves it
unchanged and the answer is `len(intervals) - kept`. Touching endpoints do not
count as an overlap, which is why the test is `s >= last_end` and not `s > last_end`.

In [10]:
from typing import List

class Solution435:
    def eraseOverlapIntervals(self, intervals: List[List[int]]) -> int:
        intervals.sort(key=lambda iv: iv[1])        # earliest finishing first
        kept, last_end = 0, float('-inf')
        for s, e in intervals:
            if s >= last_end:                       # fits after what we kept
                kept += 1
                last_end = e
        return len(intervals) - kept

ex = [[1, 2], [2, 3], [3, 4], [1, 3]]
print(f'  {ex} -> {Solution435().eraseOverlapIntervals([list(x) for x in ex])}')
room = [[s, e] for _, s, e in MEETINGS]
print(f'  the meeting room above -> remove {Solution435().eraseOverlapIntervals(room)} of {len(MEETINGS)}')

  [[1, 2], [2, 3], [3, 4], [1, 3]] -> 1
  the meeting room above -> remove 3 of 8


## 10. LeetCode 452 — Minimum Number of Arrows to Burst Balloons

Same sort, opposite comparison. Balloons are closed intervals: an arrow at
`x = 2` bursts both `[1,2]` and `[2,3]`. So a new arrow is only needed when the
next balloon starts *strictly after* the last arrow's position — `s > last`,
not `s >= last`. That single character is the whole difference between this
problem and LeetCode 435.

In [11]:
class Solution452:
    def findMinArrowShots(self, points: List[List[int]]) -> int:
        points.sort(key=lambda p: p[1])
        arrows, last = 0, float('-inf')
        for s, e in points:
            if s > last:                            # note: > not >=, touching counts as a hit
                arrows += 1
                last = e
        return arrows

balloons = [[10, 16], [2, 8], [1, 6], [7, 12]]
print(f'  {balloons} -> {Solution452().findMinArrowShots([list(b) for b in balloons])} arrows')
touching = [[1, 2], [2, 3], [3, 4], [4, 5]]
print(f'  {touching} -> {Solution452().findMinArrowShots([list(b) for b in touching])} arrows'
      ' (touching ends share an arrow)')
disjoint = [[1, 2], [3, 4], [5, 6], [7, 8]]
print(f'  {disjoint} -> {Solution452().findMinArrowShots([list(b) for b in disjoint])} arrows')

  [[10, 16], [2, 8], [1, 6], [7, 12]] -> 2 arrows
  [[1, 2], [2, 3], [3, 4], [4, 5]] -> 2 arrows (touching ends share an arrow)
  [[1, 2], [3, 4], [5, 6], [7, 8]] -> 4 arrows


## 11. LeetCode 621 — Task Scheduler

The greedy here is "always run the task with the most remaining copies that is
not cooling down", but the interesting part is that the answer can be written
without simulating anything. The most frequent task, `peak` copies of it, forces
`peak - 1` full blocks of width `n + 1`, plus one final row holding every task
tied at `peak`. If there are so many distinct tasks that the gaps fill themselves,
no idle time exists at all and the answer is just `len(tasks)`.

The simulation is kept below only as a check on the formula.

In [12]:
class Solution621:
    def leastInterval(self, tasks: List[str], n: int) -> int:
        counts = Counter(tasks)
        peak = max(counts.values())
        tied = sum(1 for c in counts.values() if c == peak)
        skeleton = (peak - 1) * (n + 1) + tied
        return max(skeleton, len(tasks))            # gaps may already be full

def least_interval_simulate(tasks, n):
    """A slow but obvious simulation, used only to check the formula above."""
    counts = Counter(tasks)
    heap = [(-c, t) for t, c in counts.items()]
    heapq.heapify(heap)
    time, cooling = 0, []                           # cooling: (ready_time, -count, task)
    while heap or cooling:
        time += 1
        while cooling and cooling[0][0] <= time:
            _, c, t = heapq.heappop(cooling)
            heapq.heappush(heap, (c, t))
        if heap:
            c, t = heapq.heappop(heap)
            if c + 1 < 0:
                heapq.heappush(cooling, (time + n + 1, c + 1, t))
    return time

for tasks, n in ((list('AAABBB'), 2), (list('AAABBB'), 0), (list('AAAAABCDEF'), 2),
                 (list('ABCDEFG'), 3), (list('AAABBBCCC'), 2)):
    f = Solution621().leastInterval(tasks, n)
    s = least_interval_simulate(tasks, n)
    print(f'  {"".join(tasks):<12} n={n} -> formula {f:>2d}, simulation {s:>2d}')

  AAABBB       n=2 -> formula  8, simulation  8
  AAABBB       n=0 -> formula  6, simulation  6
  AAAAABCDEF   n=2 -> formula 13, simulation 13
  ABCDEFG      n=3 -> formula  7, simulation  7
  AAABBBCCC    n=2 -> formula  9, simulation  9


## 12. LeetCode 1046 and 134

**1046 Last Stone Weight** is Huffman's loop with the heap turned upside down:
take the two *heaviest* stones, smash them, push the difference back. `heapq` is
a min-heap, so the stones go in negated.

**134 Gas Station** is the one greedy in this set whose proof is not an exchange
argument. If the total fuel covers the total cost a solution must exist; and if
the tank goes negative somewhere between `start` and `i`, then no station in that
stretch can be the answer either — each of them starts with even less in the tank
— so the search jumps straight to `i + 1` and the whole thing is one pass.

In [13]:
class Solution1046:
    def lastStoneWeight(self, stones: List[int]) -> int:
        heap = [-s for s in stones]                 # heapq is a min-heap, so negate
        heapq.heapify(heap)
        while len(heap) > 1:
            a = -heapq.heappop(heap)
            b = -heapq.heappop(heap)
            if a != b:
                heapq.heappush(heap, -(a - b))
        return -heap[0] if heap else 0

class Solution134:
    def canCompleteCircuit(self, gas: List[int], cost: List[int]) -> int:
        if sum(gas) < sum(cost):
            return -1                               # not enough fuel in the whole loop
        start, tank = 0, 0
        for i in range(len(gas)):
            tank += gas[i] - cost[i]
            if tank < 0:                            # i is not reachable from start
                start, tank = i + 1, 0
        return start

def can_complete_circuit_brute(gas, cost):
    """Try every start and drive the loop; O(n^2) ground truth."""
    n = len(gas)
    for start in range(n):
        tank, ok = 0, True
        for k in range(n):
            i = (start + k) % n
            tank += gas[i] - cost[i]
            if tank < 0:
                ok = False
                break
        if ok:
            return start
    return -1

print(f'  stones [2,7,4,1,8,1] -> {Solution1046().lastStoneWeight([2, 7, 4, 1, 8, 1])}')
for gas, cost in (([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]), ([2, 3, 4], [3, 4, 3])):
    print(f'  gas {gas} cost {cost} -> {Solution134().canCompleteCircuit(gas, cost)}'
          f'  (brute force says {can_complete_circuit_brute(gas, cost)})')

  stones [2,7,4,1,8,1] -> 1
  gas [1, 2, 3, 4, 5] cost [3, 4, 5, 1, 2] -> 3  (brute force says 3)
  gas [2, 3, 4] cost [3, 4, 3] -> -1  (brute force says -1)


## 13. Everything at once

If any of this ever silently changes, these asserts are what notices.

In [14]:
assert len(select_by_end(IV)) == len(select_bruteforce(IV)) == 5
assert len(select_by_start(IV)) == 3
assert len(select_by_duration(IV)) == 4
assert is_feasible(select_by_end(IV))
assert exchange_argument(IV)[1] == exchange_argument(IV)[2]
assert dp_weighted(IV, VALUES)[0] > greedy_weighted(IV, VALUES)[0]
assert is_feasible(dp_weighted(IV, VALUES)[1])
assert decode(encode(TEXT, codes), codes) == TEXT
assert decode(encode(TEXT, naive), naive) == TEXT
assert is_prefix_free(codes) and is_prefix_free(naive)
assert abs(kraft_sum(codes) - 1.0) < 1e-9
assert bits < fixed < nbits            # the broken tree is worse than not compressing
assert len(coin_greedy(COINS, 6)) == 3 and coin_dp(COINS, 6) == 2
assert Solution435().eraseOverlapIntervals([[1, 2], [2, 3], [3, 4], [1, 3]]) == 1
assert Solution452().findMinArrowShots([[10, 16], [2, 8], [1, 6], [7, 12]]) == 2
assert Solution452().findMinArrowShots([[1, 2], [3, 4], [5, 6], [7, 8]]) == 4
assert Solution1046().lastStoneWeight([2, 7, 4, 1, 8, 1]) == 1
assert Solution134().canCompleteCircuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]) == 3
assert Solution134().canCompleteCircuit([2, 3, 4], [3, 4, 3]) == -1
for tasks, n in ((list('AAABBB'), 2), (list('AAABBB'), 0), (list('AAAAABCDEF'), 2),
                 (list('ABCDEFG'), 3), (list('AAABBBCCC'), 2)):
    assert Solution621().leastInterval(tasks, n) == least_interval_simulate(tasks, n)
print('all assertions passed')

all assertions passed
